## **Chapter 2 Setting Up the Development Environment**

In [7]:
!pip install transformers torch gradio datasets

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import gradio as gr

In [9]:
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## **Chapter 3 Building a Baseline Chatbot**

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the pretrained DialoGPT model and tokenizer
MODEL_NAME = "microsoft/DialoGPT-medium"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
# Baseline chatbot function
chat_history_ids = None

def chatbot_response(user_input, chat_history_ids=None):
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")
    # Add conversational history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response


## **Chapter 4 Launch Your First Chatbot Locally**

In [ ]:
css = """
/* Container */
.container {
    background-color: #fdf4f4;
    border-radius: 15px;
    box-shadow: 0 6px 20px rgba(0, 0, 0, 0.1);
    padding: 25px;
    font-family: 'Comic Sans MS', sans-serif;
}

/* Title */
h1 {
    text-align: center;
    font-size: 32px;
    color: #ff7f7f;
    font-weight: 600;
    margin-bottom: 25px;
    font-family: 'Pacifico', sans-serif;
}

/* Outer box */
.input_output_outerbox {
    background-color: #f8d3d3; /* Light pink */
    padding: 10px;
    border-radius: 15px;
    margin-bottom: 15px;
}

/* Input and Text area */
input[type="text"], textarea {
    width: 100%;
    padding: 18px 22px;
    font-size: 18px;
    border-radius: 25px;
    border: 2px solid #ff6f61;
    background-color: #fff9e6; /* Cream color */
    color: brown;
    font-weight: bold;
    outline: none;
    transition: border-color 0.3s ease;
}

/* Keep background and text color on focus */
input[type="text"]:focus, textarea:focus {
    border-color: #ff1493;
    background-color: #fff9e6 !important;
    color: brown;
    font-weight: bold;
    box-shadow: none;
}

/* Output */
.output_text {
    padding: 16px 22px;
    background-color: #2e082e;
    border-radius: 20px;
    font-size: 18px;
    color: brown;
    font-weight: bold;
    border: 1px solid #ff6f61;
    word-wrap: break-word;
    min-height: 60px;
}

/* Button */
button {
    background-color: #ff6f61;
    color: red;
    padding: 16px 28px;
    font-size: 20px;
    font-weight: bold;
    border-radius: 30px;
    border: none;
    cursor: pointer;
    width: 100%;
    transition: background-color 0.3s ease, transform 0.2s;
}

/* Button hover effect with animation */
button:hover {
    background-color: #ff1493;
    transform: scale(1.1);
}

/* Cute footer with smaller text */
footer {
    text-align: center;
    margin-top: 20px;
    font-size: 16px;
    color: #ff6f61;
}

"""


In [ ]:
iface = gr.Interface(fn=chatbot_response,
                     theme="default",
                     inputs="text",
                     outputs="text",
                     title="Baseline Chatbot",
                     css=css)
iface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50e8c5409683a5a3cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Chapter 5 Fine-Tuning the Chatbot for Better Conversations (Most effective upgrade)**

In [ ]:
# Load a subset of DailyDialog
dataset = load_dataset("daily_dialog")
train_data = dataset["train"].shuffle(seed=42).select(range(len(dataset["train"]) // 10))
valid_data = dataset["validation"].shuffle(seed=42).select(range(len(dataset["validation"]) // 10))

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

daily_dialog.py:   0%|          | 0.00/4.85k [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Flatten multi-turn dialog structure
    text_list = [" ".join(dialog) if isinstance(dialog, list) else dialog for dialog in examples["dialog"]]

    # Tokenize each conversation
    model_inputs = tokenizer(text_list, padding="max_length", truncation=True, max_length=128)

    # Set labels = input_ids
    model_inputs["labels"] = model_inputs["input_ids"].copy()

    return model_inputs

# Tokenizing dataset
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=["dialog"])
tokenized_valid = valid_data.map(tokenize_function, batched=True, remove_columns=["dialog"])

# Convert dataset format
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_valid.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1111 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_chatbot",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    save_steps=500,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid
)

In [ ]:
# Train the model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: panzy0524 (panzy0524-linkedin) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,1.995300
1000,1.580900
1500,1.392400


TrainOutput(global_step=1668, training_loss=1.6250326947914324, metrics={'train_runtime': 684.2838, 'train_samples_per_second': 4.871, 'train_steps_per_second': 2.438, 'total_flos': 773839853715456.0, 'train_loss': 1.6250326947914324, 'epoch': 3.0})

In [ ]:
def chatbot_response(user_input):
    input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=30,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2
    )
    response = tokenizer.decode(output_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response



# Gradio UI
iface = gr.Interface(fn=chatbot_response,
                     theme="default",
                     inputs="text",
                     outputs="text",
                     title="Trained Chatbot",
                     css=css)
iface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcf6c16331cf7b9461.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Chapter 7 Further Upgrading Chatbot Responses**

### **07_01 Upgrade 1: RAG (Retrieval-Augmented Generation)**



In [ ]:
# Small knowledge base
knowledge_base = {
    "huggingface": "Hugging Face is a company specializing in Natural Language Processing technologies.",
    "transformers": "Transformers are a type of deep learning model introduced in the paper 'Attention is All You Need'.",
    "gradio": "Gradio is a Python library that allows you to rapidly create user interfaces for machine learning models."
}

def retrieve_relevant_info(query):
    # Simple keyword matching
    for keyword, info in knowledge_base.items():
        if keyword.lower() in query.lower():
            return info
    return ""

def chatbot_response(user_input):
    retrieved_info = retrieve_relevant_info(user_input)
    augmented_prompt = (retrieved_info + "\n" if retrieved_info else "") + "User: " + user_input + "\nBot:"

    input_ids = tokenizer.encode(augmented_prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.85,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.1
    )

    response = tokenizer.decode(output_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    return response.strip()


### **07_02 Upgrade 2: Improving Response Coherence and Context Awareness**

In [ ]:
conversation_history = []

def chatbot_response(user_input):
    global conversation_history
    conversation_history.append(f"User: {user_input}")
    if len(conversation_history) > 6:  # Limit to last 6 turns
        conversation_history = conversation_history[-6:]

    prompt = "\n".join(conversation_history) + "\nBot:"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.85,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.1
    )

    response = tokenizer.decode(output_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True).strip()

    conversation_history.append(f"Bot: {response}")
    return response


### **07_03 Upgrade 3: Handle Uncertain Responses with Fallback Mechanism**

In [ ]:
conversation_history = []

def chatbot_response(user_input):
    global conversation_history
    conversation_history.append(f"User: {user_input}")
    if len(conversation_history) > 6:
        conversation_history = conversation_history[-6:]

    prompt = "\n".join(conversation_history) + "\nBot:"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
        top_k=50,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(output_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True).strip()

    # Fallback if response is too short or vague
    if not response or len(response.split()) <= 2:
        response = "I'm not sure I understood that. Could you please rephrase?"

    conversation_history.append(f"Bot: {response}")
    return response
